# Notebook 15 - Object-aware foreground/background PFB + SAC

This notebook tests prompt-guided foreground style injection. It first generates the Infinity-2B baseline image, uses a lightweight CLIPSeg model to segment the main object from the content prompt and the foreground object from each style reference, then runs PFB + SAC with this schedule:

```text
scale 0,1: global style injection for overall theme
scale 3,6,9: foreground style goes to the content object, background style goes to the generated background
```

The current strong baseline, `0,1,2,9` top-1 SVD PFB + SAC with decay, is also generated for comparison.

## 1. Import project modules and build configuration

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents[:5], Path('/content/VAR_SOICT')]
VAR_SOICT_ROOT = next(
    candidate.resolve() for candidate in candidates
    if (candidate / 'src' / 'var_soict').exists()
)
sys.path.insert(0, str(VAR_SOICT_ROOT / 'src'))

from var_soict.config import ExperimentConfig
from var_soict.bootstrap import build_runtime_paths

config = ExperimentConfig(
    output_run_name='infinity2b_object_masked_prompt_segmentation',
    run_pfb_sac=False,
    run_multistep=False,
    run_top1_style_steps=True,
    run_object_masked_style_steps=True,
)
paths = build_runtime_paths(config)

print('VAR_SOICT root:', VAR_SOICT_ROOT)
print('Runtime root:', paths.runtime_root)
print('Output dir:', paths.output_dir)
print('Mask model:', config.object_mask_model_id, '| device:', config.object_mask_device)
print('Object-aware PFB steps:', config.object_masked_feature_indices)
print('Masked-only steps:', config.object_masked_mask_feature_indices)
print('Object-aware strengths:', config.object_masked_style_strength_by_step)
print('Style reference mask prompt:', config.style_reference_mask_prompt)
print('Foreground SVD rank:', config.object_masked_foreground_svd_rank, '| background SVD rank:', config.object_masked_background_svd_rank)
print('Background style strength on split steps:', config.object_masked_background_strength)


## 2. Check runtime

In [ ]:
from var_soict.bootstrap import check_torch_runtime

DEVICE = check_torch_runtime(require_cuda=True)


## 3. Install dependencies

In [ ]:
from var_soict.bootstrap import install_dependencies

install_dependencies()


## 4. Download and patch Infinity-2B GGUF assets

In [ ]:
from var_soict.bootstrap import import_gguf_loader, prepare_infinity_sources, verify_model_files

model_files = prepare_infinity_sources(config, paths)
verify_model_files(paths, model_files)
gguf_loader = import_gguf_loader(paths, model_files)


## 5. Load model components and scale schedule

In [ ]:
from var_soict.bootstrap import build_scale_schedule, load_model_bundle

bundle = load_model_bundle(config, model_files, gguf_loader)
bundle.scale_schedule = build_scale_schedule(config.model_pn, aspect_ratio=1.0)
print('Scale schedule length:', len(bundle.scale_schedule))


## 6. Sample prompts and create experiment runner

In [ ]:
from var_soict.dataset import load_random_eval_cases
from var_soict.experiment import Infinity2BExperiment
from var_soict.style_transfer import StyleTransferEngine

sessions, cases, selected_cases_csv = load_random_eval_cases(
    var_soict_root=VAR_SOICT_ROOT,
    output_dir=paths.output_dir,
    config=config,
)
engine = StyleTransferEngine(bundle, config)
experiment = Infinity2BExperiment(
    engine=engine,
    sessions=sessions,
    cases=cases,
    config=config,
    paths=paths,
)

print(f'Quantitative styles: {len(sessions)}')
print(f'Random prompts per style: {config.prompts_per_style}')
print(f'Cases per generation step: {len(cases)}')
print('Selected case manifest:', selected_cases_csv)
for session in sessions:
    print(f"- {session['style_id']} | Figure 10 #{session['figure10_index']:02d} | {session['style_label']} | {len(session['prompts'])} prompts")


## 7. Generate baseline content images

In [ ]:
baseline_results = experiment.run_baseline()


## 8. Build prompt-guided object masks from baseline images

In [ ]:
object_masks = experiment.build_prompt_object_masks()


## 9. Build foreground/background masks for style references


In [ ]:
style_reference_masks = experiment.build_style_reference_masks()


## 10. Current strong baseline: top-1 style steps 0,1,2,9

In [ ]:
top1_style_steps_results = experiment.run_top1_style_steps()


## 11. Object-aware PFB + SAC: global 0,1 and fg/bg split 3,6,9

In [ ]:
object_masked_results = experiment.run_object_masked_style_steps()


## 12. Final aggregate comparison

In [ ]:
experiment.plot_final_aggregate()


## Download outputs

In [ ]:
zip_path = experiment.download_outputs()
print('Downloaded ZIP:', zip_path)
